# Global Superstore Precleaning

This notebook prepares the Kaggle Global Superstore dataset for SQL analysis.
                                                           
I  clean dates, convert numeric fields, and validate the dataset before loading into SQL Server.


In [1]:
#Import Libraries
import numpy as np
import pandas as pd


Load Raw Data

In [2]:
#Step 1: Load the dataset
import pandas as pd

# Adjust the path if needed
df = pd.read_csv(r"C:\Users\HP\Downloads\superstore.csv", encoding="latin1")

# Quick look at the data
print(df.shape)
df.head()


(51290, 27)


,Category,City,Country,Customer.ID,Customer.Name,Discount,Market,RecordCount,Order.Date,Order.ID,...,Sales,Segment,Ship.Date,Ship.Mode,Shipping.Cost,State,Sub.Category,Year,Market2,weeknum
0,Office Supplies,Los Angeles,United States,LS-172304,Lycoris Saunders,0.0,US,1,00:00.0,CA-2011-130813,...,19,Consumer,00:00.0,Second Class,4.37,California,Paper,2011,North America,2
1,Office Supplies,Los Angeles,United States,MV-174854,Mark Van Huff,0.0,US,1,00:00.0,CA-2011-148614,...,19,Consumer,00:00.0,Standard Class,0.94,California,Paper,2011,North America,4
2,Office Supplies,Los Angeles,United States,CS-121304,Chad Sievert,0.0,US,1,00:00.0,CA-2011-118962,...,21,Consumer,00:00.0,Standard Class,1.81,California,Paper,2011,North America,32
3,Office Supplies,Los Angeles,United States,CS-121304,Chad Sievert,0.0,US,1,00:00.0,CA-2011-118962,...,111,Consumer,00:00.0,Standard Class,4.59,California,Paper,2011,North America,32
4,Office Supplies,Los Angeles,United States,AP-109154,Arthur Prichep,0.0,US,1,00:00.0,CA-2011-146969,...,6,Consumer,00:00.0,Standard Class,1.32,California,Paper,2011,North America,40


In [2]:
#Inspect the data

df.shape
df.info()
df.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51290 entries, 0 to 51289
Data columns (total 27 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Category        51290 non-null  object 
 1   City            51290 non-null  object 
 2   Country         51290 non-null  object 
 3   Customer.ID     51290 non-null  object 
 4   Customer.Name   51290 non-null  object 
 5   Discount        51290 non-null  float64
 6   Market          51290 non-null  object 
 7   RecordCount     51290 non-null  int64  
 8   Order.Date      51290 non-null  object 
 9   Order.ID        51290 non-null  object 
 10  Order.Priority  51290 non-null  object 
 11  Product.ID      51290 non-null  object 
 12  Product.Name    51290 non-null  object 
 13  Profit          51290 non-null  float64
 14  Quantity        51290 non-null  int64  
 15  Region          51290 non-null  object 
 16  Row.ID          51290 non-null  int64  
 17  Sales           51290 non-null 

,Category,City,Country,Customer.ID,Customer.Name,Discount,Market,RecordCount,Order.Date,Order.ID,...,Sales,Segment,Ship.Date,Ship.Mode,Shipping.Cost,State,Sub.Category,Year,Market2,weeknum
0,Office Supplies,Los Angeles,United States,LS-172304,Lycoris Saunders,0.0,US,1,00:00.0,CA-2011-130813,...,19,Consumer,00:00.0,Second Class,4.37,California,Paper,2011,North America,2
1,Office Supplies,Los Angeles,United States,MV-174854,Mark Van Huff,0.0,US,1,00:00.0,CA-2011-148614,...,19,Consumer,00:00.0,Standard Class,0.94,California,Paper,2011,North America,4
2,Office Supplies,Los Angeles,United States,CS-121304,Chad Sievert,0.0,US,1,00:00.0,CA-2011-118962,...,21,Consumer,00:00.0,Standard Class,1.81,California,Paper,2011,North America,32
3,Office Supplies,Los Angeles,United States,CS-121304,Chad Sievert,0.0,US,1,00:00.0,CA-2011-118962,...,111,Consumer,00:00.0,Standard Class,4.59,California,Paper,2011,North America,32
4,Office Supplies,Los Angeles,United States,AP-109154,Arthur Prichep,0.0,US,1,00:00.0,CA-2011-146969,...,6,Consumer,00:00.0,Standard Class,1.32,California,Paper,2011,North America,40


Preview of raw dataset with placeholder dates (00:00.0).

In [3]:
#Standardize column names
#Right now we have names like Customer.ID and Order.Date. Let’s make them SQL‑friendly:
df.columns = df.columns.str.strip().str.replace(r"[.\s]", "_", regex=True).str.lower()


Example:

Customer.ID → customer_id

Order.Date → order_date

In [4]:
#Copy for Cleaning 
df_clean = df.copy()


We keep a copy of the raw dataset for reference.

Fix Dates, 
Create the Clean Version with Randomized Dates

In [5]:
import numpy as np
import pandas as pd

# Define start and end dates
start_date = pd.to_datetime("2011-01-01")
end_date = pd.to_datetime("2014-12-31")
days_between = (end_date - start_date).days

# Generate random offsets for each row
random_offsets = np.random.randint(0, days_between, size=len(df_clean))

# Assign order_date by adding offsets to start_date
df_clean["order_date"] = start_date + pd.to_timedelta(random_offsets, unit="D")

# Ship dates 1–10 days after order
df_clean["ship_date"] = df_clean["order_date"] + pd.to_timedelta(
    np.random.randint(1, 10, size=len(df_clean)), unit="D"
)

# Derive year and weeknum
df_clean["year"] = df_clean["order_date"].dt.year
df_clean["weeknum"] = df_clean["order_date"].dt.isocalendar().week


Order dates randomized across 2011–2014, ship dates offset realistically.

In [6]:
#Handle missing values
# Check missing values
print(df_clean.isnull().sum())


category          0
city              0
country           0
customer_id       0
customer_name     0
discount          0
market            0
recordcount       0
order_date        0
order_id          0
order_priority    0
product_id        0
product_name      0
profit            0
quantity          0
region            0
row_id            0
sales             0
segment           0
ship_date         0
ship_mode         0
shipping_cost     0
state             0
sub_category      0
year              0
market2           0
weeknum           0
dtype: int64


In [7]:
#Convert Numeric Columns
numeric_cols = ["sales", "quantity", "discount", "profit", "shipping_cost"]
for col in numeric_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")


Ensures numeric fields are properly typed for SQL.

In [8]:
#Save cleaned dataset
df_clean.to_csv(r"C:\Users\HP\Downloads\GlobalSuperstore_clean.csv", index=False)


In [9]:
df_clean = pd.read_csv(r"C:\Users\HP\Downloads\GlobalSuperstore_clean.csv", encoding="latin1")


In [10]:
print(df_clean.shape)
df_clean.head()


(51290, 27)


,category,city,country,customer_id,customer_name,discount,market,recordcount,order_date,order_id,...,sales,segment,ship_date,ship_mode,shipping_cost,state,sub_category,year,market2,weeknum
0,Office Supplies,Los Angeles,United States,LS-172304,Lycoris Saunders,0.0,US,1,2014-05-03,CA-2011-130813,...,19,Consumer,2014-05-07,Second Class,4.37,California,Paper,2014,North America,18
1,Office Supplies,Los Angeles,United States,MV-174854,Mark Van Huff,0.0,US,1,2014-10-16,CA-2011-148614,...,19,Consumer,2014-10-18,Standard Class,0.94,California,Paper,2014,North America,42
2,Office Supplies,Los Angeles,United States,CS-121304,Chad Sievert,0.0,US,1,2014-09-25,CA-2011-118962,...,21,Consumer,2014-09-29,Standard Class,1.81,California,Paper,2014,North America,39
3,Office Supplies,Los Angeles,United States,CS-121304,Chad Sievert,0.0,US,1,2011-03-22,CA-2011-118962,...,111,Consumer,2011-03-28,Standard Class,4.59,California,Paper,2011,North America,12
4,Office Supplies,Los Angeles,United States,AP-109154,Arthur Prichep,0.0,US,1,2013-04-23,CA-2011-146969,...,6,Consumer,2013-04-30,Standard Class,1.32,California,Paper,2013,North America,17


Validation

In [11]:
print(df_clean["order_date"].min(), df_clean["order_date"].max())
print(df_clean["year"].unique())
print(df_clean["weeknum"].unique()[:10])  # sample


2011-01-01 2014-12-30
[2014 2011 2013 2012]
[18 42 39 12 17 37 50 16 20 45]


Earliest date ~2011‑01‑01, latest ~2014‑12‑30, years 2011–2014, weeks 1–52.

Validation for Shipping Delays

In [12]:
# Ensure order_date and ship_date are datetime
df_clean["order_date"] = pd.to_datetime(df_clean["order_date"])
df_clean["ship_date"] = pd.to_datetime(df_clean["ship_date"])

# Now calculate shipping delays
df_clean["shipping_delay"] = (df_clean["ship_date"] - df_clean["order_date"]).dt.days

# Quick checks
print("Min delay:", df_clean["shipping_delay"].min())

print("Max delay:", df_clean["shipping_delay"].max())
print(df_clean["shipping_delay"].value_counts().sort_index())


Min delay: 1
Max delay: 9
shipping_delay
1    5747
2    5647
3    5660
4    5632
5    5887
6    5577
7    5686
8    5710
9    5744
Name: count, dtype: int64


step 2. Create a database connection, you will put your own connection

          # Load Both Versions into SQL

Export to SQL

In [13]:
from sqlalchemy import create_engine

engine = create_engine(
    "mssql+pyodbc://DESKTOP-HRV6DMT\\SQLEXPRESS/GlobalSuperstoreDB"
    "?driver=ODBC Driver 17 for SQL Server&Trusted_Connection=yes"
)

# Raw table (all text, preserves Kaggle exactly)
df = df.astype(str)
df.to_sql("superstore_raw", con=engine, if_exists="replace", index=False)

# Clean table (randomized dates, proper datatypes)
df_clean.to_sql("superstore_clean", con=engine, if_exists="replace", index=False)


8

Clean dataset loaded into SQL Server as superstore_clean.

Automate  Pipeline